# RetailFlow - Silver Layer
Cleans, standardizes, deduplicates, and validates Bronze datasets into business-conforming Silver tables.

In [ ]:
from pathlib import Path
import pandas as pd
import re

# Dynamically resolve project root from any folder or subfolder
current = Path.cwd()
project_root = None
for candidate in [current, current / 'RetailFlow', *current.parents]:
    if (candidate / 'data' / 'raw').exists():
        project_root = candidate
        break
    if (candidate / 'RetailFlow' / 'data' / 'raw').exists():
        project_root = candidate / 'RetailFlow'
        break
if project_root is None:
    raise FileNotFoundError('RetailFlow root directory with data/raw not found.')

BRONZE_PATH = project_root / 'data' / 'bronze'
SILVER_PATH = project_root / 'data' / 'silver'
SILVER_PATH.mkdir(parents=True, exist_ok=True)

print('Bronze path:', BRONZE_PATH)
print('Silver path:', SILVER_PATH)


In [ ]:
silver_report = []

for file in sorted(BRONZE_PATH.glob('*.parquet')):
    df = pd.read_parquet(file)
    original_rows = len(df)

    # Standardize column names
    df.columns = [
        re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]+', '_', col.strip())).strip('_').lower()
        for col in df.columns
    ]

    # Remove completely empty rows
    df = df.dropna(how='all')

    # Clean text columns
    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
        df[col] = df[col].replace('', pd.NA)

    # Remove duplicate records
    df = df.drop_duplicates()

    # Save Silver dataset
    output_path = SILVER_PATH / file.name
    df.to_parquet(output_path, index=False)

    silver_report.append({
        'Dataset': file.name,
        'Original Rows': original_rows,
        'Cleaned Rows': len(df),
        'Duplicates/Empty Removed': original_rows - len(df)
    })
    print(f'Cleaned {file.name:<42} | Cleaned: {len(df):>10,} | Removed: {original_rows - len(df):>7,}')

print('\nSilver Layer Cleaning Completed!')


In [ ]:
silver_summary = pd.DataFrame(silver_report)
try:
    display(silver_summary)
except Exception:
    print(silver_summary.to_string(index=False))

print(f'\nTotal Silver datasets: {len(silver_report)}')
print(f'Total cleaned records: {silver_summary["Cleaned Rows"].sum():,}')
